<a href="https://colab.research.google.com/github/superbat3/Final_Project_ECS_2025/blob/main/testScripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y -qq gnupg2 openssl python3-pip
!pip3 install -q requests flask pgpy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../00-libpython3.10-dev_3.10.12-1~22.04.12_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.12) over (3.10.12-1~22.04.11) ...
Preparing to unpack .../01-libpython3.10_3.10.12-1~22.04.12_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.12) over (3.10.12-1~22.04.11) ...
Preparing to unpack .../02-python3.10_3.10.12-1~22.04.12_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.12) over (3.10.12-1~22.04.11) ...
Preparing to unpack .../03-libpython3.10-stdlib_3.10.12-1~22.04.12_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.12) over (3.10.12-1~22.04.11) ...
Preparing to unpack .../04-python3.10-minimal_3.10.12-1~22.04.12_amd64.deb ...
Unpacking python3.10-minimal 

In [2]:
import os, time, tempfile, threading, subprocess, statistics, shutil, socket
from pathlib import Path
from flask import Flask, request, jsonify
import requests

# --- Config
ITERATIONS = 3
SIZES = [1024, 100_000]  # small set for quick runs
PORT = 8443
HOST = "localhost"

# --- Prepare working dirs and cert
work = Path(tempfile.mkdtemp(prefix="colab_bench_"))
crt = work / "server.crt"
key = work / "server.key"
gnupghome = work / "gnupg"
os.makedirs(gnupghome, exist_ok=True)

# Generate self-signed cert
subprocess.run([
    "openssl", "req", "-x509", "-nodes", "-days", "365",
    "-newkey", "rsa:2048",
    "-keyout", str(key),
    "-out", str(crt),
    "-subj", "/CN=localhost",
    "-addext", "subjectAltName=DNS:localhost"
], check=True)


# --- Create non-interactive GPG key (no passphrase) inside GNUPGHOME
genconf = f"""%no-protection
Key-Type: RSA
Key-Length: 2048
Name-Real: Colab Test
Name-Email: test@example.com
Expire-Date: 0
"""
conf_path = work / "gpg_conf"
conf_path.write_text(genconf)
env = os.environ.copy()
env["GNUPGHOME"] = str(gnupghome)
subprocess.run(["gpg", "--batch", "--generate-key", str(conf_path)], check=True, env=env)

# --- Minimal Flask HTTPS server that saves uploads and returns processing time
app = Flask(__name__)
upload_dir = work / "uploads"
upload_dir.mkdir(exist_ok=True)

@app.route("/", methods=["GET"])
def index():
    return jsonify(status="ok"), 200

@app.route("/upload", methods=["POST"])
def upload():
    start = time.time()
    data = request.get_data()
    fname = upload_dir / str(int(start * 1000))
    fname.write_bytes(data)
    proc_ms = int((time.time() - start) * 1000)
    return jsonify(ok=True, size=len(data), processing_ms=proc_ms), 200


def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(('localhost', port))
            return False
        except socket.error as e:
            if e.errno == 98: # Address already in use
                return True
            raise e

def run_server():
    global PORT
    while is_port_in_use(PORT):
        PORT += 1
        print(f"Port {PORT-1} in use, trying {PORT}")
    print(f"Starting server on port {PORT}")
    # Flask's SSL context takes (cert, key)
    # Suppress Flask's default output to avoid clutter
    import logging
    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)
    app.run(host=HOST, port=PORT, ssl_context=(str(crt), str(key)), threaded=True)

# Start server thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for the server to be ready
max_wait_time = 30  # Increased wait time further
start_wait = time.time()
server_ready = False
while time.time() - start_wait < max_wait_time:
    try:
        # Attempt to connect to the server's root path, disable SSL verification for readiness check
        requests.get(f"https://{HOST}:{PORT}/", verify=False, timeout=1)
        server_ready = True
        print("Server is ready.")
        break
    except (requests.exceptions.ConnectionError, requests.exceptions.RequestException) as e:
        # Catching RequestException to include SSLError and other potential issues
        print(f"Waiting for server... ({type(e).__name__}: {e})")
        time.sleep(0.5) # Increased sleep time
if not server_ready:
    raise RuntimeError(f"Server did not start within {max_wait_time} seconds")


# --- Helper wrappers
def timestamp_ms():
    return int(time.time() * 1000)

def gpg_encrypt(infile, outfile, recipient="test@example.com"):
    subprocess.run([
        "gpg", "--batch", "--yes", "--trust-model", "always",
        "--output", str(outfile), "--encrypt", "--recipient", recipient, str(infile)
    ], check=True, env=env)

def gpg_decrypt(infile, outfile):
    subprocess.run([
        "gpg", "--batch", "--yes", "--trust-model", "always",
        "--output", str(outfile), "--decrypt", str(infile)
    ], check=True, env=env)

def https_upload(fname):
    with open(fname, "rb") as f:
        start = timestamp_ms()
        # Use the potentially updated PORT, keep SSL verification for the actual benchmark
        r = requests.post(f"https://{HOST}:{PORT}/upload", data=f, verify=str(crt), timeout=30)
        end = timestamp_ms()
    r.raise_for_status()
    return end - start, r.json()

# --- Benchmark loop
try:
    print("Working directory:", work)
    for size in SIZES:
        print(f"\nSize: {size} bytes")
        https_times = []
        enc_times = []
        dec_times = []

        src = work / "test.bin"
        src.write_bytes(os.urandom(size))

        for i in range(ITERATIONS):
            # HTTPS upload
            try:
                t_upload, info = https_upload(src)
                https_times.append(t_upload)
            except Exception as e:
                print("HTTPS upload failed:", e)
                continue

            # PGP encrypt/decrypt (public-key)
            enc = work / f"test_{i}.gpg"
            dec = work / f"test_{i}.dec"

            t0 = timestamp_ms()
            gpg_encrypt(src, enc)
            enc_times.append(timestamp_ms() - t0)

            t0 = timestamp_ms()
            gpg_decrypt(enc, dec)
            dec_times.append(timestamp_ms() - t0)

            # Integrity check
            if src.read_bytes() != dec.read_bytes():
                raise RuntimeError("Decrypted file does not match original")

            enc.unlink(missing_ok=True)
            dec.unlink(missing_ok=True)

        median = lambda lst: int(statistics.median(lst)) if lst else "N/A"
        print("Median HTTPS upload ms:", median(https_times))
        print("Median PGP encrypt ms:", median(enc_times))
        print("Median PGP decrypt ms:", median(dec_times))

finally:
    # Cleanup optional: comment out if you want to inspect files
    # shutil.rmtree(work)
    print("\nFinished. Artifacts in:", work)

Starting server on port 8443
Waiting for server... (ConnectionError: HTTPSConnectionPool(host='localhost', port=8443): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7c97ff1ad940>: Failed to establish a new connection: [Errno 111] Connection refused')))
 * Serving Flask app '__main__'
 * Debug mode: off


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Server is ready.
Working directory: /tmp/colab_bench_q2h_m0zh

Size: 1024 bytes
Median HTTPS upload ms: 22
Median PGP encrypt ms: 8
Median PGP decrypt ms: 21

Size: 100000 bytes
Median HTTPS upload ms: 23
Median PGP encrypt ms: 20
Median PGP decrypt ms: 25

Finished. Artifacts in: /tmp/colab_bench_q2h_m0zh


In [3]:
#HTTPS Ubtegrity Testing
import requests
from requests.exceptions import SSLError

url = "https://example.com"  # Replace with your HTTPS test server

print("=== HTTPS Integrity Test ===")

try:
    # Normal request
    response = requests.get(url, timeout=5)
    print("Original response length:", len(response.content))

    # Simulate tampering: manually corrupt the response content
    tampered_content = bytearray(response.content)
    tampered_content[10] = (tampered_content[10] + 1) % 256  # flip one byte

    # TLS/HTTPS itself would reject tampered packets in transit.
    # Here we simulate detection by comparing hashes.
    import hashlib
    original_hash = hashlib.sha256(response.content).hexdigest()
    tampered_hash = hashlib.sha256(tampered_content).hexdigest()

    print("Original SHA256:", original_hash)
    print("Tampered SHA256:", tampered_hash)
    print("Integrity check passed:", original_hash == tampered_hash)

except SSLError as e:
    print("TLS rejected tampered packet:", e)


=== HTTPS Integrity Test ===
Original response length: 513
Original SHA256: 6f5635035f36ad500b4fc4bb7816bb72ef5594e1bcae44fa074c5e988fc4c0fe
Tampered SHA256: c0d6a1b88a8fbdf1392c0f5d0e1c82eee85d50ee90454d6698eaabe383179bf9
Integrity check passed: False


In [4]:
import pgpy

# 1) Generate a signing key
key = pgpy.PGPKey.new(pgpy.constants.PubKeyAlgorithm.RSAEncryptOrSign, 2048)
uid = pgpy.PGPUID.new('Integrity Tester', email='tester@example.com')
key.add_uid(
    uid,
    usage={pgpy.constants.KeyFlags.Sign},
    hashes=[pgpy.constants.HashAlgorithm.SHA256],
    ciphers=[pgpy.constants.SymmetricKeyAlgorithm.AES256]
)

# 2) Prepare the cleartext to sign
original_text = "Integrity test message content"
tampered_text = "Integrity test message contint"  # one character changed

# 3) Create a detached signature over the raw text
signature = key.sign(original_text)  # detached signature

# 4) Verify original (should succeed) and tampered (should fail)
verified_original = key.verify(original_text, signature=signature)
verified_tampered = key.verify(tampered_text, signature=signature)

print("Original verification succeeded:", bool(verified_original))   # True
print("Tampered verification succeeded:", bool(verified_tampered))   # False


Original verification succeeded: True
Tampered verification succeeded: False


/usr/local/lib/python3.12/dist-packages/pgpy/pgp.py:2389: UserWarning: TODO: Self-sigs verification is not yet working because self-sigs are not parsed!!!
  warnings.warn("TODO: Self-sigs verification is not yet working because self-sigs are not parsed!!!")
/usr/local/lib/python3.12/dist-packages/pgpy/pgp.py:2406: UserWarning: TODO: Revocation checks are not yet implemented!!!
  warnings.warn("TODO: Revocation checks are not yet implemented!!!")
/usr/local/lib/python3.12/dist-packages/pgpy/pgp.py:2407: UserWarning: TODO: Flags (s.a. `disabled`) checks are not yet implemented!!!
  warnings.warn("TODO: Flags (s.a. `disabled`) checks are not yet implemented!!!")
